# Research Question 3, Text Extension: Sourcing, Building, and Modeling County-Level News Sentiment

This notebook documents the full text/sentiment data pipeline used to extend the
Research Question 3 demographic model with issue-based news tone:

1. **Testing candidate text/sentiment data sources** — screening survey microdata,
   social platforms, and news archives against one hard requirement: the ability to
   resolve to Pennsylvania county-level (or at least sub-state) geography.
2. **Processing the data** — building the county-level news-tone measure from the
   source that passed screening (GDELT), including the media-market aggregation
   used to solve a county-level data-sparsity problem.
3. **Completing the text-based analysis** — adding the four resulting tone variables
   to the six-variable demographic Ridge regression from the earlier Q3 notebook,
   for a ten-predictor model.

Each section starts with a markdown cell explaining *why* that step exists before the
code that performs it.

## Part 1 — Testing Candidate Text/Sentiment Data Sources

### 1.1 The requirement

The original hypothesis (H3) called for issue-based sentiment drawn from open-ended
survey text. Before building any pipeline, every candidate source was tested against
one specific requirement: **can it resolve to Pennsylvania county-level, or at least
sub-state, geography?** A source with a strong topical match but no usable geography
is not a viable predictor for a county-level model, regardless of its academic
pedigree or data quality on other dimensions.

The cells below reconstruct the screening process as data — first the general-purpose
sources (social platforms, alternative news APIs, search-interest data), then a closer
look at the two academic survey panels (Pew ATP, ANES), then a live feasibility check
against the source that ultimately passed: the GDELT news archive.

In [ ]:
import pandas as pd

# Sources screened before settling on GDELT. Verdicts are the result of documentation
# review and, for GDELT, a live feasibility check (Section 1.3 below) -- not all of
# these have an API that could be probed directly (several are account-gated).
alt_sources = pd.DataFrame([
    ("Twitter/X", "Full-archive historical search, priced per call.",
     "Ruled out - geotagging was largely removed from the default posting flow years "
     "ago; the small, self-selecting share of posts that remain geotagged skews "
     "younger, more urban, and more politically online than the general electorate."),
    ("Media Cloud", "Free academic platform (UMass Amherst/Northeastern) built for "
     "local-news research, with a place-tagging engine designed to reduce national-"
     "wire noise.",
     "Not pursued - fully account-gated, with no anonymous querying; registration "
     "requires a written justification and email confirmation before any query can "
     "be tested."),
    ("Local-outlet targeting (NELA-Local method)", "Restrict queries to each county's "
     "own local paper's domain (e.g., the Bedford Gazette for Bedford County) rather "
     "than broad keyword search.",
     "Promising but labor-intensive - would require identifying and validating "
     "roughly 67 outlet domains by hand; the reference dataset only covers "
     "April 2020-December 2021."),
    ("Google Trends", "Search-interest-by-region for issue keywords.",
     "Ruled out - geographic granularity is capped at the metro/media-market level, "
     "the same resolution the GDELT aggregation already uses, so it offers no "
     "improvement."),
    ("Reddit", "County- and city-specific subreddits.",
     "Ruled out - most small Pennsylvania counties have no dedicated subreddit, and "
     "the user base that exists skews young and urban."),
    ("Meta Content Library", "Public-content API spanning Facebook, Instagram, and "
     "Threads, including Facebook Groups.",
     "Ruled out for this project - restricted to academic/nonprofit use with an "
     "ICPSR-style vetting process taking four to six weeks, plus a setup and monthly "
     "fee under at least one access path."),
], columns=["source", "what_it_offers", "verdict"])

pd.set_option("display.max_colwidth", 100)
alt_sources

### 1.2 Revisiting survey sources: Pew ATP and ANES

Two established public-opinion sources were evaluated as a completeness check on the
news-based approach: the **Pew Research Center American Trends Panel (ATP)**, a
probability-based national panel of roughly 10,000 U.S. adults with dedicated
2024-cycle waves on immigration, abortion, cultural issues, and top-problems-facing-
the-country; and the **American National Election Studies (ANES) 2024 Time Series
Study**, the discipline's longest-running election survey, with 5,521 pre-election
completions.

Both hit the same geographic-resolution limit that already shaped the news-based
approach, just at different tiers — ATP has no state variable at all in its public
release, while ANES has a state identifier but redacts county FIPS/name/tract behind
a restricted-use ICPSR agreement. This comparison is captured directly (not via a live
query — both are documentation-based access-tier facts, not something an anonymous
API call can confirm).

In [ ]:
survey_comparison = pd.DataFrame({
    "": [
        "Topic overlap with GDELT issue set",
        "State identifier (public release)",
        "County identifier",
        "Access effort",
        "Usable Pennsylvania sample size",
    ],
    "Pew ATP": [
        "Strong", "No", "No, at any access tier",
        "Low - free account, instant download",
        "Unknown, likely small",
    ],
    "ANES 2024": [
        "Strong", "Yes", "Restricted-use only, via ICPSR",
        "Low (public file) / high (restricted geography, multi-week ICPSR process)",
        "~100-250 statewide; thinner still per county",
    ],
})
survey_comparison

### 1.3 Live feasibility check: GDELT at the county level

The first tractable substitute tested was the [GDELT Project](https://www.gdeltproject.org/)'s
public news archive — a free, keyless API that can be filtered by exact date range and
keyword, with article-level tone scores (`mode=tonechart`) that aggregate into an
issue-level sentiment measure.

Before committing to any aggregation strategy, a direct county-level feasibility check
was run against one large county (Erie, ~270,000 residents) and two small ones
(Bedford, ~48,000; Forest, ~7,000 — the state's smallest), across the four target
issues (crime, immigration, abortion, economy). The query pattern below — a
county-name anchor plus issue keywords, restricted to the Jan 1 – Nov 5, 2024 window —
is the same pattern used for the full media-market run in Part 2, just applied to a
single county instead of a market of several counties.

In [ ]:
import requests
import time
import urllib.parse

GDELT_BASE = "https://api.gdeltproject.org/api/v2/doc/doc"
START, END = "20240101000000", "20241105000000"

test_counties = {
    "Erie":    "Erie Pennsylvania",       # large county
    "Bedford": "Bedford Pennsylvania",    # small county
    "Forest":  "Forest Pennsylvania",     # smallest PA county
}

issues = {
    "crime": "(crime OR violence OR shooting)",
    "immigration": '(immigration OR migrants OR "border crossing")',
    "abortion": '(abortion OR "reproductive rights")',
    "economy": "(economy OR inflation OR jobs)",
}

def fetch_tonechart(query: str) -> dict:
    """Call the GDELT tonechart endpoint and return total article count + weighted
    average tone for one query. Returns (None, None) on request/parse failure."""
    params = {
        "query": query, "mode": "tonechart",
        "startdatetime": START, "enddatetime": END, "format": "json",
    }
    url = GDELT_BASE + "?" + urllib.parse.urlencode(params)
    try:
        resp = requests.get(url, timeout=20)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        return None, None, str(e)
    tonechart = data.get("tonechart", [])
    total = sum(b.get("count", 0) for b in tonechart)
    avg_tone = (sum(b.get("bin", 0) * b.get("count", 0) for b in tonechart) / total
                if total > 0 else None)
    return total, avg_tone, None

feasibility_rows = []
for county, anchor in test_counties.items():
    for issue, kw in issues.items():
        count, tone, err = fetch_tonechart(f"{anchor} {kw}")
        feasibility_rows.append({"county": county, "issue": issue,
                                  "article_count": count, "avg_tone": tone, "error": err})
        time.sleep(1)  # be polite to the free, keyless public API

feasibility_df = pd.DataFrame(feasibility_rows)
feasibility_df

**What this confirmed:** the API worked as advertised, but exposed a coverage
problem. Erie (large county) returned a usable number of articles per issue, while
Bedford and Forest — both small/rural counties — returned very few articles, and
several of Forest's hits were unrelated national wire stories caught by loose keyword
matching rather than genuine local reporting. With more than 40 of Pennsylvania's 67
counties in this small/rural category, a *county-level* version of this measure would
be mostly noise.

This is the sparsity problem that motivates the **media-market aggregation** strategy
built in Part 2: instead of querying each of the 67 counties individually, queries are
built once per Nielsen media market (11 markets covering all of Pennsylvania) so each
query pools enough population to return a stable article count.

In [ ]:
# Quick view of the sparsity pattern that motivated aggregating to media markets
# rather than querying every county individually
summary = feasibility_df.groupby("county")["article_count"].agg(["sum", "count"])
summary.columns = ["total_articles_across_4_issues", "issues_queried"]
summary

## Part 2 — Processing the Data: Building the County-Level News-Tone Measure

### 2.1 Build the query plan: 11 media markets x 4 issues

Pennsylvania's 67 counties fall into 11 Nielsen media markets (DMAs). Each market's
query anchor combines the market's distinguishing place names — for markets that
straddle a state line, the anchor uses PA-specific county/city names so the search
still resolves to Pennsylvania coverage, not the whole multi-state market.

In [ ]:
markets = {
    "Buffalo_NY_PA": '("McKean County" OR "Potter County") Pennsylvania',
    "Elmira_NY_PA": '"Tioga County" Pennsylvania',
    "Erie_PA": '(Erie OR "Crawford County" OR "Warren County") Pennsylvania',
    "Harrisburg_Lancaster_York": '(Harrisburg OR Lancaster OR York) Pennsylvania',
    "Johnstown_Altoona": '(Johnstown OR Altoona OR "State College") Pennsylvania',
    "New_York_NY_PA": '"Pike County" Pennsylvania',
    "Philadelphia": "Philadelphia Pennsylvania",
    "Pittsburgh": "Pittsburgh Pennsylvania",
    "Washington_DC_PA": '("Franklin County" OR "Fulton County" OR Chambersburg) Pennsylvania',
    "Wilkes_Barre_Scranton": '("Wilkes-Barre" OR Scranton) Pennsylvania',
    "Youngstown_OH_PA": '"Mercer County" Pennsylvania',
}

# Same four issues tested in the Part 1 feasibility check
query_plan = [
    {"market": mkt, "issue": issue, "query": f"{anchor} {kw}"}
    for mkt, anchor in markets.items()
    for issue, kw in issues.items()
]
print(f"{len(query_plan)} queries planned ({len(markets)} markets x {len(issues)} issues)")
pd.DataFrame(query_plan).head()

### 2.2 Fetch and parse each market/issue tonechart

Same `fetch_tonechart` helper from the Part 1 feasibility check, run once per
market/issue pair. Results are stored in long format (one row per market-issue
combination) — this is the natural shape for a pivot into wide format later, and
matches how the results were logged during the original run.

In [ ]:
tone_rows = []
for q in query_plan:
    count, tone, err = fetch_tonechart(q["query"])
    tone_rows.append({"market": q["market"], "issue": q["issue"],
                       "article_count": count, "avg_tone": tone})
    time.sleep(1)

tone_long = pd.DataFrame(tone_rows)
tone_long.to_csv("gdelt_market_tone.csv", index=False)
print(f"Saved gdelt_market_tone.csv: {tone_long.shape}")
tone_long.head(10)

### 2.3 Known parsing bug and reliability threshold

Two data-quality issues were caught and handled before this data was usable:

1. **A confirmed parsing bug** — the initial Erie/economy query was mis-recorded as 0
   articles because of a URL-encoding edge case in the original request. It was
   re-verified directly against the GDELT API (1,067 articles, weighted average tone
   -1.6570) and hard-coded as a correction here, since re-querying a fixed historical
   date range should reproduce the same figure.
2. **A reliability threshold** — Part 1's feasibility check showed that very small
   article counts produce noisy, unreliable tone averages (a handful of articles can
   swing the average wildly). Any market/issue combination with fewer than 15 articles
   is treated as missing rather than trusted, and filled in later by imputation
   (Section 2.5).

In [ ]:
import numpy as np

# Re-load in case Section 2.2 was skipped and gdelt_market_tone.csv already exists
tone_long = pd.read_csv("gdelt_market_tone.csv")

# --- Confirmed bug fix: Erie/economy mis-parsed as 0 articles ---
mask = (tone_long["market"] == "Erie_PA") & (tone_long["issue"] == "economy")
tone_long.loc[mask, "avg_tone"] = -1.6570
tone_long.loc[mask, "article_count"] = 1067

# --- Reliability threshold: fewer than 15 articles is treated as missing ---
MIN_ARTICLES = 15
tone_long["reliable_tone"] = np.where(
    tone_long["article_count"] >= MIN_ARTICLES, tone_long["avg_tone"], np.nan
)

# --- Reshape long -> wide: one row per market, one column per issue ---
wide = tone_long.pivot(index="market", columns="issue", values="reliable_tone")
wide.columns = [f"{c}_tone" for c in wide.columns]
wide = wide.reset_index()

print("Market-level tone (NaN = insufficient coverage, <15 articles):")
wide

### 2.4 Map GDELT market keys to the county-to-DMA crosswalk

The market names used for the GDELT queries (e.g. `"Erie_PA"`) are convenience labels,
not the official Nielsen DMA names used in the county crosswalk file
(`pa_dma_raw.csv`, sourced from [A. Patton's US-TVDMA-BY-COUNTY crosswalk](https://github.com/alex-patton/US-TVDMA-BY-COUNTY)).
This step maps one to the other, then attaches each market's tone scores to every
county assigned to that market — this is the core of the media-market aggregation:
every county in a market receives the *same* tone score, since the underlying
sentiment measure can only resolve to market-level geography.

In [ ]:
market_to_dma = {
    "Buffalo_NY_PA": "Buffalo, NY - PA DMA",
    "Elmira_NY_PA": "Elmira, NY - PA DMA",
    "Erie_PA": "Erie, PA DMA",
    "Harrisburg_Lancaster_York": "Harrisburg - Lancaster - Lebanon - York, PA DMA",
    "Johnstown_Altoona": "Johnstown - Altoona, PA DMA",
    "New_York_NY_PA": "New York, NY - CT - NJ - PA DMA",
    "Philadelphia": "Philadelphia, PA - DE - NJ DMA",
    "Pittsburgh": "Pittsburgh, PA - MD- WV DMA",
    "Washington_DC_PA": "Washington, DC - MD - PA - VA - WV DMA",
    "Wilkes_Barre_Scranton": "Wilkes Barre - Scranton, PA DMA",
    "Youngstown_OH_PA": "Youngstown, OH - PA DMA",
}
wide["TVDMA"] = wide["market"].map(market_to_dma)
assert wide["TVDMA"].isna().sum() == 0, "unmapped market name"

county_market = pd.read_csv("pa_dma_raw.csv")  # STATE, COUNTY, TVDMA, ... (67 PA counties)
county_tone = county_market.merge(wide, on="TVDMA", how="left")
assert len(county_tone) == 67, f"expected 67 counties, got {len(county_tone)}"

issue_cols = ["crime_tone", "immigration_tone", "abortion_tone", "economy_tone"]
county_tone[["COUNTY", "TVDMA"] + issue_cols].head(10)

### 2.5 Impute any remaining gaps

A handful of markets still fall short of the 15-article reliability threshold even
after aggregating to market level (mostly the smallest, most rural media markets).
Rather than dropping those counties from the model, their tone values are imputed with
the **cross-market median** for that issue, and every affected county is flagged with
`text_data_imputed = 1` so the imputation is visible and auditable rather than silent —
this flag can be used later as a robustness check (e.g., re-running the model excluding
imputed counties).

In [ ]:
imputed_flag = county_tone[issue_cols].isna().any(axis=1)
print("Counties needing imputation (insufficient local news coverage):")
print(county_tone.loc[imputed_flag, "COUNTY"].tolist())

for c in issue_cols:
    county_tone[c] = county_tone[c].fillna(county_tone[c].median())

county_tone["text_data_imputed"] = imputed_flag.astype(int)
county_tone[["COUNTY"] + issue_cols + ["text_data_imputed"]].head(10)

### 2.6 Merge into the master election + demographic dataset

The final step joins the county-level tone measure onto the same 67-county file used
for the demographics-only model (`election_demo_merged_q3.csv`, itself the output of
an earlier election-results + ACS demographics merge on county FIPS code). County
names are normalized (title case, stripped whitespace) before merging on name, since
the tone crosswalk file uses uppercase county names while the election file uses
mixed case. Two asserts guard the result: the row count must stay at 67, and no county
should be missing a tone value after imputation.

In [ ]:
election_demo = pd.read_csv("election_demo_merged_q3.csv")

county_tone["county_norm"] = county_tone["COUNTY"].str.title().str.strip()
election_demo["county_norm"] = election_demo["county"].str.title().str.strip()

merged = election_demo.merge(
    county_tone[["county_norm", "TVDMA"] + issue_cols + ["text_data_imputed"]],
    on="county_norm", how="left",
)

missing = merged[merged["crime_tone"].isna()]
print("Unmatched counties (should be none):", missing["county"].tolist())
assert len(merged) == 67
assert merged["crime_tone"].isna().sum() == 0

merged = merged.drop(columns=["county_norm"])
merged.to_csv("election_demo_text_merged_q3.csv", index=False)

print(f"\nSaved election_demo_text_merged_q3.csv: {merged.shape}")
print("\nCounties with imputed (low-confidence) text data:",
      merged.loc[merged["text_data_imputed"] == 1, "county"].tolist())
merged[["county", "TVDMA"] + issue_cols + ["text_data_imputed"]].head(8)

## Part 3 — Completing the Text-Based Analysis: Ten-Predictor Ridge Regression

### 3.1 Model overview

This extends the six-variable demographic Ridge model from the earlier Q3 notebook
with the four county-level news-tone variables built in Part 2, for ten predictors
total:

**Demographic (6):** `pct_bachelor_plus`, `median_hh_income`, `pct_white`,
`pct_poverty`, `pct_unemployed`, `pct_foreignborn`

**Text/sentiment (4):** `crime_tone`, `immigration_tone`, `abortion_tone`,
`economy_tone` (lower/more negative = more negative press coverage on that issue)

The dependent variable is unchanged: `demvoteshare_change` (Democratic share of the
two-party vote, 2020 to 2024, by Pennsylvania county). The pipeline is identical to
the demographics-only analysis: load, VIF diagnostic, standardize, RidgeCV with
leave-one-out cross-validation (LOOCV), then report standardized coefficients,
in-sample R², and the honest out-of-sample LOOCV R²/RMSE.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

MERGED_FILE = "election_demo_text_merged_q3.csv"
OUTCOME = "demvoteshare_change"

DEMO_VARS = [
    "pct_bachelor_plus", "median_hh_income", "pct_white",
    "pct_poverty", "pct_unemployed", "pct_foreignborn",
]
TEXT_VARS = ["crime_tone", "immigration_tone", "abortion_tone", "economy_tone"]
PREDICTORS = DEMO_VARS + TEXT_VARS

ALPHA_GRID = np.logspace(-2, 4, 100)  # log-spaced grid for RidgeCV to search

df = pd.read_csv(MERGED_FILE)
missing_cols = [c for c in PREDICTORS + [OUTCOME] if c not in df.columns]
if missing_cols:
    raise SystemExit(f"Missing columns in {MERGED_FILE}: {missing_cols}")
print(f"Loaded {len(df)} rows from {MERGED_FILE}.")

### 3.2 Assemble the modeling dataset

Keeps only the county identifier, outcome, and the ten predictors, then drops any rows
with missing values (reporting which counties, if any, were dropped, so a data problem
fails visibly rather than silently).

In [ ]:
df_model = df[["county", OUTCOME] + PREDICTORS].dropna()
n_dropped = len(df) - len(df_model)
if n_dropped:
    dropped = df.loc[~df.index.isin(df_model.index), "county"].tolist()
    print(f"Dropped {n_dropped} rows with missing values: {dropped}")
else:
    print("No missing values - all counties retained.")

X_raw = df_model[PREDICTORS]
y = df_model[OUTCOME].values
print(f"\nModeling dataset: {len(df_model)} counties, {len(PREDICTORS)} predictors.")
df_model.head()

### 3.3 Multicollinearity diagnostic (VIF)

Computed on the raw (unstandardized) predictors. Income, education, and poverty are
already known to be structurally correlated at the county level (the same finding from
the demographics-only model) — the question here is whether adding the four
news-tone variables introduces *additional* collinearity, or whether they behave as a
largely independent block. Either way, elevated VIFs are the direct motivation for
Ridge over OLS at this sample size.

In [ ]:
def compute_vif(X: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "variable": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    }).sort_values("VIF", ascending=False)

vif_table = compute_vif(X_raw)
print(vif_table.to_string(index=False))
print("\nRule of thumb: VIF > 5-10 signals meaningful multicollinearity.")

### 3.4 Standardize predictors

Ridge's penalty term is scale-dependent, so all ten predictors are z-scored (mean 0,
standard deviation 1) before fitting. This also puts demographic percentages, income
(in dollars), and tone scores (roughly -5 to +5) on a common scale, so their
coefficients are directly comparable in the results below.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
X = pd.DataFrame(X, columns=PREDICTORS, index=X_raw.index)
X.describe()

### 3.5 Fit RidgeCV with leave-one-out cross-validation

`RidgeCV` searches the 100-point log-spaced alpha grid and picks whichever value
minimizes prediction error under leave-one-out CV (LOOCV) — more stable than standard
k-fold at this sample size (n = 67 counties, or fewer if any were dropped in
Section 3.2).

In [ ]:
loo = LeaveOneOut()
ridge_cv = RidgeCV(alphas=ALPHA_GRID, cv=loo, scoring="neg_mean_squared_error")
ridge_cv.fit(X, y)

print(f"Selected alpha (lambda): {ridge_cv.alpha_:.4f}")

### 3.6 Standardized coefficients

Because every predictor was z-scored in Section 3.4, these coefficients are directly
comparable in magnitude — the largest-magnitude coefficient is the strongest
association with vote-share change, net of the other nine predictors, regardless of
sign.

In [ ]:
coef_table = pd.DataFrame({
    "variable": PREDICTORS,
    "standardized_coef": ridge_cv.coef_,
}).sort_values("standardized_coef", key=abs, ascending=False)

coef_table

### 3.7 Model fit: in-sample vs. honest out-of-sample (LOOCV)

In-sample R² reflects how well the model fits data it already saw during training and
will run higher than the true generalization performance — especially now that the
predictor count has grown from six to ten on the same 67 observations. LOOCV re-fits
the model once per county, each time predicting a county the model never saw, which is
the primary number to report given the small sample size.

In [ ]:
y_pred_in = ridge_cv.predict(X)
ss_tot = np.sum((y - y.mean()) ** 2)
r2_in = 1 - np.sum((y - y_pred_in) ** 2) / ss_tot

y_pred_loo = cross_val_predict(RidgeCV(alphas=[ridge_cv.alpha_]), X, y, cv=loo)
r2_loo = 1 - np.sum((y - y_pred_loo) ** 2) / ss_tot
rmse_loo = np.sqrt(np.mean((y - y_pred_loo) ** 2))

print(f"In-sample R^2:             {r2_in:.3f}")
print(f"LOOCV R^2 (out-of-sample): {r2_loo:.3f}")
print(f"LOOCV RMSE:                {rmse_loo:.5f}")

### 3.8 Coefficient chart

Horizontal bar chart of all ten standardized coefficients, ranked by magnitude. Teal
marks a positive association with Democratic vote-share change; rust marks a negative
one — the same color convention used in the demographics-only chart, so the two are
visually comparable side by side.

In [ ]:
label_map = {
    "pct_unemployed": "Unemployment rate",
    "pct_foreignborn": "Foreign-born share",
    "pct_poverty": "Poverty rate",
    "pct_bachelor_plus": "Bachelor's degree+",
    "median_hh_income": "Median household income",
    "pct_white": "White population share",
    "crime_tone": "Crime coverage tone",
    "immigration_tone": "Immigration coverage tone",
    "abortion_tone": "Abortion coverage tone",
    "economy_tone": "Economy coverage tone",
}
chart_df = coef_table.copy()
chart_df["label"] = chart_df["variable"].map(label_map)
chart_df = chart_df.sort_values("standardized_coef", key=abs, ascending=True)
colors = ["#A84B2F" if v < 0 else "#20808D" for v in chart_df["standardized_coef"]]

fig = go.Figure(go.Bar(
    x=chart_df["standardized_coef"], y=chart_df["label"], orientation="h",
    marker_color=colors,
    text=[f"{v:+.4f}" for v in chart_df["standardized_coef"]], textposition="outside",
))
fig.update_layout(
    title="Standardized Ridge coefficients: demographic and news-tone predictors<br>of Democratic vote-share change",
    xaxis_title="Standardized coefficient",
    template="plotly_white", height=480, width=800, showlegend=False,
)
fig.add_vline(x=0, line_color="#D4D1CA")
fig.show()

### 3.9 Save outputs

Per-county predictions and the standardized coefficient table are saved to CSV for use
in the write-up's tables and figures — the same output pattern as the demographics-only
model, so both can be loaded and compared side by side.

In [ ]:
df_model = df_model.copy()
df_model["ridge_predicted"] = y_pred_in
df_model.to_csv("ridge_q3_full10_predictions.csv", index=False)
coef_table.to_csv("ridge_q3_full10_coefficients.csv", index=False)

print("Saved: ridge_q3_full10_predictions.csv, ridge_q3_full10_coefficients.csv")